# BCI demo export (for the exhibition robot demo)
This notebook trains EEGNet for two "new users" and exports everything the browser demo needs.

**Honest design (so the demo is not a trick):**
- For each demo user (subject 1 and subject 2), the model is trained on the OTHER 8 subjects only.
  The demo user is a person the model has never seen.
- Same pipeline as the study: 8 channels, 8-30 Hz bandpass, Euclidean Alignment, 3 classes.
- Alignment uses only the first 48 unlabeled trials of session 1. The demo trials come from session 2.
- Demo trials are picked at RANDOM (8 per class), not cherry-picked. The accuracy on all 216 session-2
  trials is saved too, so the page can show the real number.
- The model is exported with batch-norm folded into the convolutions, so a small JavaScript
  function can run it in the browser. This notebook checks that the exported weights give the same
  answers as PyTorch.

Output: `demo_s1.json`, `demo_s2.json`, and `demo_export.zip`. They are also copied to your Drive
folder `BCI_demo_export`. Run with a T4 GPU. It takes about 10 to 15 minutes (most of it is the data download).

In [ ]:
!pip install -q moabb mne torch scikit-learn scipy

In [ ]:
import os, json, base64, copy, time, zipfile
import numpy as np
import torch, torch.nn as nn
from scipy.signal import butter, filtfilt
from numpy.lib.stride_tricks import sliding_window_view

DATA = "real"                 # "real" or "synthetic" (synthetic is only for testing the code)
PRETRAIN_EPOCHS = 60
DEMO_SUBJECTS = [1, 2]        # held-out users: one model per user
SHOWN_PER_CLASS = 8
N_REF = 48
SEED = 0
CLASSES = ("left_hand", "right_hand", "feet")
CHANNELS = ("FC3", "FC1", "FCz", "FC2", "FC4", "C3", "Cz", "C4")
ALL_SUBJECTS = list(range(1, 10))
PRETRAIN_LR, BATCH = 1e-3, 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = "demo_export"
os.makedirs(OUT_DIR, exist_ok=True)
print("device:", DEVICE)

In [ ]:
class EEGNet(nn.Module):
    def __init__(self, n_classes=3, n_channels=8, n_samples=1000,
                 F1=8, D=2, F2=16, kernel_length=125, dropout_rate=0.5):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, kernel_length), padding=(0, kernel_length // 2), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(), nn.AvgPool2d((1, 4)), nn.Dropout(dropout_rate))
        self.block2 = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, (1, 16), padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(), nn.AvgPool2d((1, 8)), nn.Dropout(dropout_rate))
        with torch.no_grad():
            flat = self.block2(self.block1(torch.zeros(1, 1, n_channels, n_samples))).numel()
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(flat, n_classes))
        self.cfg = dict(F1=F1, D=D, F2=F2, K=kernel_length, C=n_channels, T=n_samples, n_classes=n_classes)

    def forward(self, x):
        return self.classifier(self.block2(self.block1(x)))

In [ ]:
def load_real(subjects):
    import moabb.datasets as mdatasets
    from moabb.paradigms import MotorImagery
    ds = mdatasets.BNCI2014_001() if hasattr(mdatasets, "BNCI2014_001") else mdatasets.BNCI2014001()
    paradigm = MotorImagery(channels=list(CHANNELS), events=list(CLASSES), n_classes=len(CLASSES))
    ep, _, _ = paradigm.get_data(dataset=ds, subjects=[subjects[0]], return_epochs=True)
    ch_names = list(ep.ch_names)
    label_map = {c: i for i, c in enumerate(CLASSES)}
    data = {}
    for sid in subjects:
        X, labels, meta = paradigm.get_data(dataset=ds, subjects=[sid])
        y = np.array([label_map[l] for l in labels], dtype=np.int64)
        names = list(dict.fromkeys(meta["session"].tolist()))
        assert len(names) == 2, names
        sess = np.array([names.index(s) for s in meta["session"].tolist()])
        data[sid] = dict(X=X.astype(np.float32), y=y, sess=sess)
        print(f"subject {sid}: {X.shape}, trials per session {np.bincount(sess)}")
    return data, ch_names


def make_synthetic(subjects, per_class_per_session=50, n_ch=8, n_samples=1001, fs=250, seed=0):
    rng = np.random.RandomState(seed)
    t = np.arange(n_samples) / fs
    class_channel = {0: 5, 1: 7, 2: 6}
    data = {}
    for sid in subjects:
        Xs, ys, ss = [], [], []
        strength = 0.8 + 1.2 * rng.rand()
        for s in (0, 1):
            for c in range(3):
                for _ in range(per_class_per_session):
                    x = rng.randn(n_ch, n_samples).astype(np.float32)
                    x[class_channel[c]] += strength * np.sin(2 * np.pi * 10 * t + rng.rand() * 6.28)
                    Xs.append(x); ys.append(c); ss.append(s)
        Xs, ys, ss = np.stack(Xs), np.array(ys), np.array(ss)
        order = np.concatenate([rng.permutation(np.where(ss == s)[0]) for s in (0, 1)])
        data[sid] = dict(X=Xs[order], y=ys[order], sess=ss[order])
    return data, list(CHANNELS)


RAW, CH_NAMES = load_real(ALL_SUBJECTS) if DATA == "real" else make_synthetic(ALL_SUBJECTS)
print("channel order in the data:", CH_NAMES)
SHOW_IDX = [CH_NAMES.index(c) for c in ("C3", "Cz", "C4")]

In [ ]:
def bandpass(X, low=8.0, high=30.0, fs=250.0, order=4):
    nyq = fs / 2.0
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    return filtfilt(b, a, X, axis=-1).astype(np.float32)


def zscore_from_ref(X, Xref):
    m = Xref.mean(axis=(0, 2), keepdims=True)
    s = Xref.std(axis=(0, 2), keepdims=True) + 1e-8
    return ((X - m) / s).astype(np.float32)


def ea_matrix(Xref):
    R = np.mean([x @ x.T / x.shape[1] for x in Xref], axis=0)
    w, V = np.linalg.eigh(R)
    return (V @ np.diag(np.clip(w, 1e-10, None) ** -0.5) @ V.T).astype(np.float32)


def apply_ea(X, Xref):
    return np.einsum("ij,njt->nit", ea_matrix(Xref), X).astype(np.float32)


FILT = {s: dict(X=bandpass(d["X"]), y=d["y"], sess=d["sess"]) for s, d in RAW.items()}
SRC_EA, TGT_EA, TGT_SHOW = {}, {}, {}
for sid, d in FILT.items():
    s0 = np.where(d["sess"] == 0)[0]
    ref = d["X"][s0[:min(N_REF, len(s0))]]
    SRC_EA[sid] = apply_ea(d["X"], d["X"])          # as a training subject: all its data
    TGT_EA[sid] = apply_ea(d["X"], ref)             # as the demo user: only 48 unlabeled setup trials
    TGT_SHOW[sid] = zscore_from_ref(d["X"], ref)[:, SHOW_IDX, :]   # what the page draws (C3, Cz, C4)
print("preprocessing done")

In [ ]:
def pretrain(Xs, ys, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = EEGNet(len(CLASSES), Xs.shape[1], Xs.shape[2]).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=PRETRAIN_LR)
    crit = nn.CrossEntropyLoss()
    Xt = torch.from_numpy(Xs).unsqueeze(1).to(DEVICE)
    yt = torch.from_numpy(ys).to(DEVICE)
    n = len(yt)
    for ep in range(PRETRAIN_EPOCHS):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, BATCH):
            idx = perm[i:i + BATCH]
            if len(idx) < 2:
                continue
            opt.zero_grad(); crit(model(Xt[idx]), yt[idx]).backward(); opt.step()
        if (ep + 1) % 10 == 0:
            print(f"    epoch {ep+1}/{PRETRAIN_EPOCHS}")
    return model.eval()


@torch.no_grad()
def probs_torch(model, X, chunk=64):
    model.eval(); out = []
    for i in range(0, len(X), chunk):
        xb = torch.from_numpy(X[i:i + chunk]).unsqueeze(1).to(DEVICE)
        out.append(torch.softmax(model(xb), 1).cpu().numpy())
    return np.concatenate(out)

In [ ]:
# ---------- Export: fold batch-norm into the convolutions, then a NumPy forward pass ----------
def fold(conv_w, bn):
    s = (bn.weight / torch.sqrt(bn.running_var + bn.eps)).detach().cpu()
    shift = (bn.bias.detach().cpu() - bn.running_mean.detach().cpu() * s)
    return conv_w * s.view(-1, *([1] * (conv_w.dim() - 1))), shift


def export_weights(model):
    cfg = model.cfg
    b1, b2 = model.block1, model.block2
    w1, bb1 = fold(b1[0].weight.detach().cpu(), b1[1])
    w2, bb2 = fold(b1[2].weight.detach().cpu(), b1[3])
    wd = b2[0].weight.detach().cpu()
    wp, bp = fold(b2[1].weight.detach().cpu(), b2[2])
    lin = model.classifier[1]
    P = dict(
        w1=w1.squeeze().numpy(), b1=bb1.detach().numpy(),            # (F1, K), (F1,)
        w2=w2.squeeze().numpy(), b2=bb2.detach().numpy(),            # (F1*D, C), (F1*D,)
        wd=wd.squeeze().numpy(),                                     # (F1*D, 16)
        wp=wp.squeeze().numpy(), bp=bp.detach().numpy(),             # (F2, F1*D), (F2,)
        wl=lin.weight.detach().cpu().numpy(), bl=lin.bias.detach().cpu().numpy())
    return cfg, {k: np.asarray(v, dtype=np.float32) for k, v in P.items()}


def elu(x):
    return np.where(x > 0, x, np.exp(np.minimum(x, 0)) - 1)


def forward_np(x, cfg, P):
    """x: (C, T). The same steps will be written in JavaScript."""
    F1, D, K = cfg["F1"], cfg["D"], cfg["K"]
    pad = K // 2
    win = sliding_window_view(np.pad(x, ((0, 0), (pad, pad))), K, axis=1)       # (C, T, K)
    y1 = np.einsum("fk,ctk->fct", P["w1"], win) + P["b1"][:, None, None]        # (F1, C, T)
    f_of = np.arange(F1 * D) // D
    y2 = np.einsum("oc,oct->ot", P["w2"], y1[f_of]) + P["b2"][:, None]          # (F1*D, T)
    y2 = elu(y2)
    T2 = y2.shape[1] // 4
    y2 = y2[:, :T2 * 4].reshape(F1 * D, T2, 4).mean(2)
    win2 = sliding_window_view(np.pad(y2, ((0, 0), (8, 8))), 16, axis=1)        # (F1*D, T2+1, 16)
    y3 = np.einsum("ok,otk->ot", P["wd"], win2)
    y4 = elu(np.einsum("po,ot->pt", P["wp"], y3) + P["bp"][:, None])
    T3 = y4.shape[1] // 8
    y4 = y4[:, :T3 * 8].reshape(-1, T3, 8).mean(2)
    logits = P["wl"] @ y4.reshape(-1) + P["bl"]
    e = np.exp(logits - logits.max())
    return e / e.sum()


def quant(a):
    scale = float(np.abs(a).max()) / 32767.0
    q = np.round(a / scale).astype(np.int16)
    return base64.b64encode(q.tobytes()).decode(), scale


def r6(a):
    return np.round(np.asarray(a, dtype=np.float64), 6).tolist()

In [ ]:
paths = []
for s in DEMO_SUBJECTS:
    print(f"\n=== Demo user: subject {s} (training on the other 8 subjects) ===")
    src = [x for x in ALL_SUBJECTS if x != s]
    Xs = np.concatenate([SRC_EA[x] for x in src])
    ys = np.concatenate([FILT[x]["y"] for x in src])
    model = pretrain(Xs, ys, SEED)

    d = FILT[s]
    s1 = np.where(d["sess"] == 1)[0]
    Xs1, ys1 = TGT_EA[s][s1], d["y"][s1]
    p_all = probs_torch(model, Xs1)
    acc_full = float((p_all.argmax(1) == ys1).mean())
    print(f"  accuracy on all {len(ys1)} session-2 trials: {acc_full*100:.1f}%")

    rng = np.random.RandomState(7)
    pick = np.concatenate([rng.choice(np.where(ys1 == c)[0], SHOWN_PER_CLASS, replace=False) for c in range(len(CLASSES))])
    pick = rng.permutation(pick)
    xm, ysh, xshow = Xs1[pick], ys1[pick], TGT_SHOW[s][s1][pick]

    xm_b64, xm_scale = quant(xm)
    xs_b64, xs_scale = quant(xshow)
    xm_deq = (np.round(xm / xm_scale) * xm_scale).astype(np.float32)   # what the browser will really see

    cfg, P = export_weights(model)
    ref = np.stack([forward_np(x, cfg, P) for x in xm_deq])
    torch_ref = probs_torch(model, xm_deq)
    diff = float(np.abs(ref - torch_ref).max())
    same_pred = bool((ref.argmax(1) == torch_ref.argmax(1)).all())
    print(f"  export check: max |numpy - torch| = {diff:.2e}, same predictions: {same_pred}")
    assert diff < 1e-3 and same_pred, "exported model does not match PyTorch"

    acc_shown = float((ref.argmax(1) == ysh).mean())
    payload = dict(
        meta=dict(subject=int(s), classes=["Left hand", "Right hand", "Feet"], fs=250,
                  channels_model=CH_NAMES, channels_shown=["C3", "Cz", "C4"],
                  accuracy_session2_all=acc_full, n_session2_trials=int(len(ys1)),
                  accuracy_shown_trials=acc_shown, n_shown=int(len(pick)),
                  trained_on_subjects=src, preprocessing="bandpass 8-30 Hz, Euclidean alignment (48 unlabeled setup trials)",
                  dataset="BCI Competition IV 2a (BNCI2014-001), loaded with MOABB; 3 classes: left hand, right hand, feet",
                  note="Recorded data replayed offline. Model never saw this user. Trials chosen at random."),
        model=dict(cfg=cfg, **{k: r6(v) for k, v in P.items()}),
        trials=dict(y=ysh.tolist(), probs_ref=r6(ref),
                    x_model_b64=xm_b64, x_model_scale=xm_scale, x_model_shape=list(xm.shape),
                    x_show_b64=xs_b64, x_show_scale=xs_scale, x_show_shape=list(xshow.shape)))
    path = f"{OUT_DIR}/demo_s{s}.json"
    json.dump(payload, open(path, "w"))
    paths.append(path)
    print(f"  saved {path} ({os.path.getsize(path)/1e6:.2f} MB); accuracy on the {len(pick)} shown trials: {acc_shown*100:.1f}%")

with zipfile.ZipFile("demo_export.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        z.write(p, os.path.basename(p))
print("\nzip ready: demo_export.zip")

In [ ]:
# ---------- Save to Drive and download ----------
try:
    from google.colab import drive, files
    drive.mount("/content/drive")
    dest = "/content/drive/MyDrive/BCI_demo_export"
    os.makedirs(dest, exist_ok=True)
    for p in paths + ["demo_export.zip"]:
        os.system(f"cp {p} {dest}/")
    print("copied to Drive folder BCI_demo_export")
    files.download("demo_export.zip")
except ImportError:
    print("not running in Colab; files are in", OUT_DIR)